# Stage 2 — Multi-label CNN classifier (Colab)

Standalone notebook for training the Stage 2 CNN on the crop dataset produced by Chapter 10 of `01_main_pipeline.ipynb`.

**Why a separate Colab notebook?** Stage 2 is built with Keras / TensorFlow. TensorFlow does not support GPU on native Windows from version 2.11 onwards, so training on the project workstation would fall back to CPU and take many hours. Colab provides a free GPU (T4 or L4) and the training takes minutes.

**Three-stage methodology** (following Unit 8 of the course):

| Stage | What | Models | Epochs |
|-------|------|--------|--------|
| 1. Ablation | Compare two architectures on equal footing, with the backbone **frozen** (feature extraction) | VGG16 + ResNet50 | 20 each |
| 2. Refinement | Take the winner from stage 1 and **fine-tune** its top layers with a very low learning rate | ResNet50 only | +15 |
| 3. Final evaluation | Run the fine-tuned model on the held-out test sets | ResNet50 fine-tuned | — (in `01_main_pipeline.ipynb` Chapter 12) |

**Workflow:**
1. Upload `stage2_crops.zip` to `MyDrive/PPE_Project/` on your Google Drive.
2. Open this notebook in Colab and select **Runtime → Change runtime type → GPU**.
3. Run all cells. Cells that find an existing trained model in Drive will skip retraining.
4. All artifacts (`best.keras`, `history.csv`, plots) end up in `MyDrive/PPE_Project/stage2_results/`.
5. Sync the results back to the laptop and commit them under `results/stage2/` in the repo.


## Before you run

1. **Upload `stage2_crops.zip` to Drive** under `MyDrive/PPE_Project/`. The zip is ~210 MB.
2. **Change runtime to GPU**: `Runtime → Change runtime type → Hardware accelerator: T4 GPU (or L4 GPU)`.
3. **Run all cells** (`Runtime → Run all`).


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT      = "/content/drive/MyDrive/PPE_Project"
ZIP_PATH        = f"{DRIVE_ROOT}/stage2_crops.zip"
RESULTS_DIR     = f"{DRIVE_ROOT}/stage2_results"
EXTRACT_TO      = "/content/stage2_crops"

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(ZIP_PATH), f"Missing: {ZIP_PATH}. Upload stage2_crops.zip to {DRIVE_ROOT} first."
print(f"Found zip: {ZIP_PATH}")
print(f"Results will go to: {RESULTS_DIR}")


In [ ]:
# Extract zip to local /content (fast SSD; much faster than reading from Drive).
# We extract manually because PowerShell's Compress-Archive on Windows writes
# entry names with backslashes; Python's z.extractall would then create files
# like "train\crops\image.jpg" in the root instead of real subdirectories.
import zipfile, os, shutil, time

if os.path.exists(EXTRACT_TO) and os.path.exists(f"{EXTRACT_TO}/train/labels.csv"):
    print(f"Already extracted at {EXTRACT_TO} - skipping.")
else:
    if os.path.exists(EXTRACT_TO):
        shutil.rmtree(EXTRACT_TO)
    os.makedirs(EXTRACT_TO)

    print(f"Extracting {ZIP_PATH} -> {EXTRACT_TO} (with path normalization)...")
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        for info in z.infolist():
            # Normalize backslashes to forward slashes (PowerShell ZIP quirk)
            name = info.filename.replace("\\", "/")
            if name.endswith("/"):
                continue
            target = os.path.join(EXTRACT_TO, name)
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)
    print(f"Done in {time.time() - t0:.1f}s")

# Verify each split has crops/ and labels.csv
import pandas as pd
for split in ("train", "val", "test_in_domain", "test_out_of_domain"):
    csv = f"{EXTRACT_TO}/{split}/labels.csv"
    crops_dir = f"{EXTRACT_TO}/{split}/crops"
    n_csv = sum(1 for _ in open(csv)) - 1
    n_files = len(os.listdir(crops_dir))
    print(f"  {split:<22s} csv={n_csv:>6,}  files={n_files:>6,}  match={n_csv == n_files}")


In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TF version          : {tf.__version__}")
print(f"GPU available       : {bool(tf.config.list_physical_devices('GPU'))}")
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    print(f"  - {g}")


In [ ]:
# Hyperparameters
IMG_SIZE     = 224
BATCH_SIZE   = 32
EPOCHS       = 20
INITIAL_LR   = 1e-4

# Feature-extraction (frozen backbone). Per Unit 8 of the course.
FREEZE_BACKBONE = True


In [ ]:
def load_split(split_name, shuffle=False):
    """Load a split into a tf.data.Dataset of (image, [helmet, vest])."""
    base = f"{EXTRACT_TO}/{split_name}"
    df = pd.read_csv(f"{base}/labels.csv")
    paths  = [f"{base}/crops/{fn}" for fn in df["filename"]]
    labels = df[["helmet", "vest"]].values.astype("float32")

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        return img, label

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=min(2000, len(df)), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, len(df)


train_ds, n_train = load_split("train", shuffle=True)
val_ds,   n_val   = load_split("val",   shuffle=False)

print(f"Train : {n_train:,} crops  ({n_train // BATCH_SIZE} batches)")
print(f"Val   : {n_val:,} crops  ({(n_val + BATCH_SIZE - 1) // BATCH_SIZE} batches)")


In [ ]:
# Augmentation block (applied only at training time)
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(0.05, seed=SEED),
    layers.RandomZoom(0.1, seed=SEED),
    layers.RandomBrightness(0.2, seed=SEED),
    layers.RandomContrast(0.2, seed=SEED),
], name="augmentation")


def build_classifier(backbone_fn, preprocess_fn, name):
    """Multi-label classifier: frozen ImageNet backbone + small Dense head."""
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)
    x = preprocess_fn(x)

    backbone = backbone_fn(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    backbone.trainable = not FREEZE_BACKBONE
    x = backbone(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(2, activation="sigmoid", name="helmet_vest")(x)

    model = keras.Model(inputs, outputs, name=name)
    return model


def train_one(backbone_fn, preprocess_fn, name):
    """Compile, train, and save weights + history. Returns history dict.
    Skips retraining if a saved best.keras already exists in Drive."""
    out_dir       = f"{RESULTS_DIR}/{name}"
    os.makedirs(out_dir, exist_ok=True)
    weights_path  = f"{out_dir}/best.keras"
    history_path  = f"{out_dir}/history.csv"

    if os.path.exists(weights_path) and os.path.exists(history_path):
        print(f"\n[skip] {name}: existing weights found at {weights_path}")
        return pd.read_csv(history_path).to_dict("list")

    print(f"\n{'='*60}\nTraining {name}\n{'='*60}")

    model = build_classifier(backbone_fn, preprocess_fn, name)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

    cbs = [
        keras.callbacks.ModelCheckpoint(weights_path, save_best_only=True,
                                        monitor="val_auc", mode="max"),
        keras.callbacks.EarlyStopping(patience=8, monitor="val_auc", mode="max",
                                      restore_best_weights=True),
        keras.callbacks.CSVLogger(history_path),
    ]

    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS, callbacks=cbs, verbose=2,
    )

    print(f"Best weights -> {weights_path}")
    print(f"History      -> {history_path}")
    return history.history


In [ ]:
vgg_hist = train_one(
    applications.VGG16,
    applications.vgg16.preprocess_input,
    "vgg16",
)


In [ ]:
resnet_hist = train_one(
    applications.ResNet50,
    applications.resnet50.preprocess_input,
    "resnet50",
)


In [ ]:
# Ablation plot: VGG16 vs ResNet50 (both at feature-extraction stage)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for hist, name, color in [(vgg_hist, "VGG16", "#1e40af"),
                          (resnet_hist, "ResNet50", "#dc2626")]:
    epochs = range(1, len(hist["loss"]) + 1)
    axes[0].plot(epochs, hist["loss"],     color=color, linestyle="-",  label=f"{name} train")
    axes[0].plot(epochs, hist["val_loss"], color=color, linestyle="--", label=f"{name} val")

    axes[1].plot(epochs, hist["auc"],     color=color, linestyle="-",  label=f"{name} train")
    axes[1].plot(epochs, hist["val_auc"], color=color, linestyle="--", label=f"{name} val")

    axes[2].plot(epochs, hist["accuracy"],     color=color, linestyle="-",  label=f"{name} train")
    axes[2].plot(epochs, hist["val_accuracy"], color=color, linestyle="--", label=f"{name} val")

for ax, title, ylabel in zip(axes,
                              ["Loss (BCE)", "AUC", "Accuracy"],
                              ["loss", "AUC", "accuracy"]):
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Stage 2.1 (Ablation) - VGG16 vs ResNet50 (feature extraction, frozen backbone)",
             fontsize=12, y=1.02)
plt.tight_layout()
plot_path = f"{RESULTS_DIR}/comparison.png"
plt.savefig(plot_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"\nSaved ablation plot to: {plot_path}")


In [ ]:
# =============================================================================
# Stage 2.2 - Fine-tune ResNet50 (the winner from the ablation)
# =============================================================================
# Unfreeze the top 30 layers of the ResNet50 backbone and continue training
# with a 10x lower learning rate to refine the model.

FT_NAME       = "resnet50_finetuned"
FT_OUT_DIR    = f"{RESULTS_DIR}/{FT_NAME}"
os.makedirs(FT_OUT_DIR, exist_ok=True)
FT_WEIGHTS    = f"{FT_OUT_DIR}/best.keras"
FT_HISTORY    = f"{FT_OUT_DIR}/history.csv"

FT_EPOCHS         = 15
FT_LR             = 1e-5     # 10x smaller than feature-extraction lr
UNFREEZE_TOP_N    = 30       # last 30 layers of the backbone

if os.path.exists(FT_WEIGHTS) and os.path.exists(FT_HISTORY):
    print(f"[skip] {FT_NAME}: existing weights found at {FT_WEIGHTS}")
    ft_hist = pd.read_csv(FT_HISTORY).to_dict("list")
else:
    print(f"\n{'='*60}\nFine-tuning ResNet50\n{'='*60}")

    # Load the feature-extraction winner
    fe_path = f"{RESULTS_DIR}/resnet50/best.keras"
    print(f"  Loading FE checkpoint: {fe_path}")
    ft_model = keras.models.load_model(fe_path)

    # Locate the ResNet50 sub-model inside our wrapper
    backbone = None
    for layer in ft_model.layers:
        if "resnet" in layer.name.lower():
            backbone = layer
            break
    if backbone is None:
        raise RuntimeError("Could not find ResNet50 backbone in the loaded model")

    # Unfreeze the backbone, then re-freeze all but the top N layers
    backbone.trainable = True
    for layer in backbone.layers[:-UNFREEZE_TOP_N]:
        layer.trainable = False

    trainable_params = int(np.sum([np.prod(w.shape) for w in ft_model.trainable_weights]))
    print(f"  Backbone has {len(backbone.layers)} layers; unfreezing the top {UNFREEZE_TOP_N}")
    print(f"  Trainable parameters: {trainable_params:,}")

    # Re-compile with low learning rate
    ft_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=FT_LR),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

    ft_cbs = [
        keras.callbacks.ModelCheckpoint(FT_WEIGHTS, save_best_only=True,
                                        monitor="val_auc", mode="max"),
        keras.callbacks.EarlyStopping(patience=5, monitor="val_auc", mode="max",
                                      restore_best_weights=True),
        keras.callbacks.CSVLogger(FT_HISTORY),
    ]

    print(f"  Fine-tuning {FT_EPOCHS} epochs at lr={FT_LR} ...")
    ft_history = ft_model.fit(
        train_ds, validation_data=val_ds,
        epochs=FT_EPOCHS, callbacks=ft_cbs, verbose=2,
    )
    ft_hist = ft_history.history

    print(f"  Best weights -> {FT_WEIGHTS}")
    print(f"  History      -> {FT_HISTORY}")


In [ ]:
# =============================================================================
# Plot the fine-tuning trajectory and compare to the FE phase
# =============================================================================
# Concatenate ResNet50 FE history + ResNet50 FT history so the x-axis spans
# both phases. A vertical line marks where fine-tuning began.

fe_len = len(resnet_hist["loss"])
ft_len = len(ft_hist["loss"])
total_epochs = fe_len + ft_len

def cat(metric):
    return list(resnet_hist[metric]) + list(ft_hist[metric])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
x_all = range(1, total_epochs + 1)

for metric_pair, ax, title in [
    (("loss", "val_loss"),         axes[0], "Loss (BCE)"),
    (("auc", "val_auc"),           axes[1], "AUC"),
    (("accuracy", "val_accuracy"), axes[2], "Accuracy"),
]:
    train_metric, val_metric = metric_pair
    ax.plot(x_all, cat(train_metric), color="#dc2626", linestyle="-",  label="train")
    ax.plot(x_all, cat(val_metric),   color="#dc2626", linestyle="--", label="val")
    ax.axvline(x=fe_len + 0.5, color="black", linestyle=":", alpha=0.7,
               label=f"FT starts (epoch {fe_len + 1})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Stage 2.2 (Refinement) - ResNet50: feature extraction -> fine-tuning",
             fontsize=12, y=1.02)
plt.tight_layout()
ft_plot_path = f"{RESULTS_DIR}/finetune_trajectory.png"
plt.savefig(ft_plot_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"\nSaved fine-tune plot to: {ft_plot_path}")


In [ ]:
# Final summary across all three Stage 2 phases
print("=" * 78)
print("Stage 2 - validation metrics across all three phases (best epoch each)")
print("=" * 78)
print(f"{'Phase':<32s} {'loss':>8s} {'acc':>8s} {'AUC':>8s} {'P':>8s} {'R':>8s}")
print("-" * 78)

for hist, name in [
    (vgg_hist,    "VGG16          (FE, ablation)"),
    (resnet_hist, "ResNet50       (FE, ablation)"),
    (ft_hist,     "ResNet50       (FT, refined) "),
]:
    best_idx = int(np.argmax(hist["val_auc"]))
    print(
        f"{name:<32s} "
        f"{hist['val_loss'][best_idx]:>8.4f} "
        f"{hist['val_accuracy'][best_idx]:>8.4f} "
        f"{hist['val_auc'][best_idx]:>8.4f} "
        f"{hist['val_precision'][best_idx]:>8.4f} "
        f"{hist['val_recall'][best_idx]:>8.4f}"
    )

print()
print(f"All artifacts under: {RESULTS_DIR}")
print(f"  vgg16/best.keras                  <- ablation: VGG16 (feature extraction)")
print(f"  vgg16/history.csv")
print(f"  resnet50/best.keras               <- ablation: ResNet50 (feature extraction)")
print(f"  resnet50/history.csv")
print(f"  resnet50_finetuned/best.keras     <- FINAL MODEL: ResNet50 fine-tuned")
print(f"  resnet50_finetuned/history.csv")
print(f"  comparison.png                    <- ablation plot")
print(f"  finetune_trajectory.png           <- FE -> FT trajectory")


## Stage 2.3 — Final evaluation on the two test sets

This is the headline experiment of Stage 2. We take the fine-tuned ResNet50 (the final model from Stage 2.2) and run it once on each of the two held-out test sets:

- **Test In-Domain** — Kaggle CSS test crops (155 person crops). Same domain the model was trained on.
- **Test Out-of-Domain** — Ultralytics dataset crops (2,243 person crops). A domain the model has never seen.

The gap between the two numbers is the **Stage 2 Cross-Dataset Gap** — the empirical answer to *"does the classifier generalize beyond the training domain?"*.

Per the project protocol the test sets are touched exactly once, here.


In [ ]:
# Prepare the two test datasets (no shuffle - deterministic eval)
test_in_domain_ds, n_in_domain = load_split("test_in_domain",     shuffle=False)
test_ood_ds,       n_ood       = load_split("test_out_of_domain", shuffle=False)

print(f"Test In-Domain      : {n_in_domain:,} crops")
print(f"Test Out-of-Domain  : {n_ood:,} crops")


In [ ]:
# Load the fine-tuned model (Stage 2.2 winner) and evaluate on each test set
print(f"Loading final model: {FT_WEIGHTS}")
final_model = keras.models.load_model(FT_WEIGHTS)

print("\n" + "=" * 60)
print("Test In-Domain - Kaggle CSS test crops")
print("=" * 60)
in_domain_eval = final_model.evaluate(test_in_domain_ds, return_dict=True, verbose=2)

print("\n" + "=" * 60)
print("Test Out-of-Domain - Ultralytics crops")
print("=" * 60)
ood_eval = final_model.evaluate(test_ood_ds, return_dict=True, verbose=2)

print("\nRaw metrics:")
print("  In-Domain     :", {k: round(v, 4) for k, v in in_domain_eval.items()})
print("  Out-of-Domain :", {k: round(v, 4) for k, v in ood_eval.items()})


In [ ]:
# Pull val metrics from the FT history (best-AUC epoch) for a 3-row comparison
best_val_idx = int(np.argmax(ft_hist["val_auc"]))

eval_summary = pd.DataFrame([
    {
        "Split":     "Validation (Kaggle val)",
        "Crops":     n_val,
        "loss":      ft_hist["val_loss"][best_val_idx],
        "accuracy":  ft_hist["val_accuracy"][best_val_idx],
        "AUC":       ft_hist["val_auc"][best_val_idx],
        "precision": ft_hist["val_precision"][best_val_idx],
        "recall":    ft_hist["val_recall"][best_val_idx],
    },
    {
        "Split":     "Test In-Domain (Kaggle test)",
        "Crops":     n_in_domain,
        "loss":      in_domain_eval["loss"],
        "accuracy":  in_domain_eval["accuracy"],
        "AUC":       in_domain_eval["auc"],
        "precision": in_domain_eval["precision"],
        "recall":    in_domain_eval["recall"],
    },
    {
        "Split":     "Test Out-of-Domain (Ultralytics)",
        "Crops":     n_ood,
        "loss":      ood_eval["loss"],
        "accuracy":  ood_eval["accuracy"],
        "AUC":       ood_eval["auc"],
        "precision": ood_eval["precision"],
        "recall":    ood_eval["recall"],
    },
])

formatters = {
    "Crops":     lambda v: f"{v:,}",
    "loss":      lambda v: f"{v:.4f}",
    "accuracy":  lambda v: f"{v:.4f}",
    "AUC":       lambda v: f"{v:.4f}",
    "precision": lambda v: f"{v:.4f}",
    "recall":    lambda v: f"{v:.4f}",
}
print(eval_summary.to_string(index=False, formatters=formatters))

# Cross-Dataset Gap (In-Domain - Out-of-Domain)
gap = {m: eval_summary.loc[1, m] - eval_summary.loc[2, m]
       for m in ("AUC", "precision", "recall", "accuracy")}

print()
print("=" * 60)
print("Cross-Dataset Gap  =  In-Domain  -  Out-of-Domain")
print("=" * 60)
for m, v in gap.items():
    print(f"  {m:<12s}: {v:+.4f}")

# Save
EVAL_DIR = f"{RESULTS_DIR}/evaluation"
os.makedirs(EVAL_DIR, exist_ok=True)
eval_summary.to_csv(f"{EVAL_DIR}/summary.csv", index=False)
print(f"\nSaved CSV  to: {EVAL_DIR}/summary.csv")


In [ ]:
# Bar chart: AUC / Precision / Recall / Accuracy across val + 2 test sets
metrics_to_plot = ["AUC", "precision", "recall", "accuracy"]
short_labels    = ["Val\n(Kaggle val)", "Test In-Domain\n(Kaggle test)", "Test Out-of-Domain\n(Ultralytics)"]
colors          = ["#94a3b8", "#1e40af", "#dc2626"]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, metric in zip(axes, metrics_to_plot):
    values = eval_summary[metric].values
    bars = ax.bar(short_labels, values, color=colors, edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_title(metric, fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", labelsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Stage 2 - ResNet50 fine-tuned - val vs both test sets", fontsize=12, y=1.02)
plt.tight_layout()
plot_path = f"{EVAL_DIR}/comparison.png"
plt.savefig(plot_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved plot to: {plot_path}")


## Next steps

1. **All artifacts are now in Drive** under `MyDrive/PPE_Project/stage2_results/`:
   - `vgg16/`, `resnet50/`, `resnet50_finetuned/` — the three trained models
   - `evaluation/summary.csv` and `evaluation/comparison.png` — Stage 2.3 final eval
   - `comparison.png` and `finetune_trajectory.png` — diagnostic plots

2. **Sync the small files (CSVs + PNGs) to the laptop** and commit them under `results/stage2/`. The fine-tuned best.keras stays in Drive (200 MB, too large for GitHub).

3. **Update Chapter 12 in `01_main_pipeline.ipynb`** with the numbers from `evaluation/summary.csv`. The main notebook will summarise the Stage 2 story without re-running TF code.
